---
title: Dataset Construction
date: 09/2025
authors:
  - name: James Butler
    affiliations: ucb
  - name: Michelle Maclennan
    affiliation: bas
  - name: Fernando Pérez
    affiliation: ucb
  - name: Jon McAuliffe
    affiliation: ucb
affiliations:
  - id: ucb
    institution: University of California Berkeley
    ror: https://ror.org/01an7q238
    department: Statistics
  - id: bas
    institution: British Antarctic Survey
    ror: https://ror.org/01rhff309
---

This notebook uses cloud-friendly workflows originally detailed [here](https://jbbutler.github.io/antarctic_AR_dataset/dataset-construction/) to create a tabular dataset of relevant characteristic and impact quantities associated with each identified AR. Specifically, for each AR event, we stream the relevant days of MERRA-2 data for each variable of interest.

In this notebook, we build a full dataset of quantities for each AR. Not all quantities will be used in the modelling stages but are nonetheless useful for identifying and further investigating individual events as part of exploratory data analyses.

:::{important}
If running the streaming workflows on CryoCloud, make sure to choose a server with enough RAM, as well as CPUs for parallelization. In the streaming loops, the RAM used by this notebook should not exceed 40GB, so any server that satisfies this requirement should be ok. Of course, this depends on how many cores you use to parallelize..

We opt for the ~119 GB RAM servers with ~15 CPUs available, parallelizing our streaming loops over 10 CPUs. From start to finish, this notebook took about 12 hours to fully execute, using [jupyter-keepalive](https://github.com/minrk/jupyter-keepalive) to keep the server alive in the background. This package is included in the provided `environment.yml` file for this project.
:::

## Setup

First, we'll load up any packages and modules we might need.

:::{note}
If using the `environment.yml` provided to create a conda environment or build a custom image on CryoCloud, all of these packages, including `artools`, should be installed.
:::

In [1]:
# load external packages
import os
import pandas as pd
import xarray as xr
from pathlib import Path
import earthaccess
import ray
from tqdm import tqdm
from huggingface_hub import login, HfApi
import logging
import sys

In [2]:
# load artools
import artools
from artools.loading_utils import load_ais, load_cell_areas, EarthdataGatekeeper, load_catalog
from artools.display_utils import display_catalog
from artools.attribute_utils import *
from artools.compute_attributes_streaming import *

/home/jovyan/extreme_antarctic_ARs/notebooks


Let's also load up some useful constants. Some of these are explained further into the notebook.

In [3]:
OUTPUT_DIR = '../outputs/data_products'
NUM_CPUS = 15
CHUNK_SIZE = 10
BATCH_SIZE = 15

Now, we log in to HuggingFace so we can grab the catalogs and any other auxiliary data products.

In [4]:
login()

Next, let's load up the catalog.

In [5]:
# load the catalog
storms = load_catalog('epsspace0.5_epstime12_minpts5_nreppts10_seed12345.h5')
# take only those that are landfalling
storms = storms[storms.is_landfalling]

Finally, we load up a mask of grid cells for Antarctica, as well as a mapping of grid cell to cell area. We will need these to compute quantities over the Antarctic ice sheet, or any quantities expressed over areas.

In [6]:
ais_mask = load_ais()
cell_areas = load_cell_areas()

## Computing Attributes and Impacts

In this section, we begin computing storm attributes and impacts. We first begin with quantities associated with the geographic details of the storm and its landfall, including the where and when. Then, we add reanalysis-derived quantities, including cumulative precipitation, maximum integrated water vapor, etc.

### Geographic Quantities

Let's start with computing geographic attributes of each landfalling storm. This includes the following quantities:
+ `max_area`: the largest areal extent of the storm over its complete lifetime
+ `mean_landfalling_area`: the average landfalling areal extent of the storm
+ `cumulative_landfalling_area`: the cumulative time x space the storm spent over the ice sheet
+ `duration`: how long did it last
+ `start_date`: when did the storm first appear
+ `end_date`: when did it dissipate
+ `max_south_extent`: how far south did it go, in degrees latitude?
+ `region`: the region of Antarctica it made landfall in (either `West`, `East 1`, or `East 2`)

Helper functions to compute each of these attributes given a storm's `xarray.DataArray` can be found in the `attributes_utils` module of `artools`.

In [7]:
storms['max_area'] = storms['data_array'].apply(lambda x: compute_max_area(x, cell_areas)).astype(int)
storms['mean_landfalling_area'] = storms['data_array'].apply(lambda x: compute_mean_area(x, cell_areas, ais_mask)).astype(int)
storms['cumulative_landfalling_area'] = storms['data_array'].apply(lambda x: compute_cumulative_spacetime(x, cell_areas, ais_mask)).astype(int)
storms['duration'] = storms['data_array'].apply(compute_duration)
storms['start_date'] = storms['data_array'].apply(add_start_date)
storms['end_date'] = storms['data_array'].apply(add_end_date)
storms['max_south_extent'] = storms['data_array'].apply(compute_max_southward_extent)

In [8]:
# regions defined by ranges of longitudes
region_defs = {'West': [-150, -30], 
               'East 1': [-30, 75],
               'East 2': [75, -150]}
region_masks = find_region_masks(region_defs, ais_mask)
storms['region'] = storms['data_array'].apply(lambda x: find_landfalling_region(x, cell_areas, region_masks))

In [9]:
storms.to_hdf('../outputs/data_products/storms_setting.h5', key='df')

/tmp/ipykernel_3511/3596746951.py:1: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block5_values] [items->Index(['data_array'], dtype='str')]

  storms.to_hdf('../outputs/data_products/storms_setting.h5', key='df')


### Atmospheric Quantities

Now for the more interesting part, let's compute atmospheric characteristics and attributes for each storm. This includes things like their landfalling moisture content, wind speeds, as well as their impacts, like cumulative snowfall on the ice sheet or maximum surface temperature anomaly.

:::{attention}
To access MERRA-2 data in the cloud, you'll need to first create a NASA Earthdata account and create the relevant [Earthdata prerequisite files](https://github.com/nasa/gesdisc-tutorials/blob/main/notebooks/How_to_Generate_Earthdata_Prerequisite_Files.ipynb), most notably the `.netrc` file with your Earthdata login credentials. You will also need to be in the AWS `us-west-2` region to stream from s3. If you are running this notebook on CryoCloud, you are in `us-west-2`.
:::

To carry out our analysis, we will need to stream the following variables from MERRA-2:
+ 2m temperature anomaly
+ Integrated water vapor (IWV)
+ Sea-level pressure (SLP)
+ Poleward integrated vapor transport (vIVT)
+ Snowfall
+ Rainfall
+ 850 hPa poleward wind
+ Omega

These variables are contained in one of the datasets listed below in the dictionary of dataset DOIs.

In [10]:
data_dois = {'climatology': '10.5067/5ESKGQTZG7FO',
             'T2M_TQV_SLP': '10.5067/3Z173KIE2TPD',
             'VFLXQV_PRECIP': '10.5067/Q5GVUVUIVGO7',
             'V850': '10.5067/VJAFPLI1CSIV',
             'OMEGA': '10.5067/QBZ6MG944HW0'}

#### Climatology

**Dataset**: [MERRA-2 instM_2d_asm_Nx: 2d,Monthly mean,Single-Level,Assimilation,Single-Level Diagnostics V5.12.4 (M2IMNXASM)](https://disc.gsfc.nasa.gov/datasets/M2IMNXASM_5.12.4/summary?keywords=10.5067%2F5ESKGQTZG7FO)

Since some of the characteristics we want to compute are anomalies, such as the maximum landfalling 2m temperature anomaly underneath the footprint of the AR, as well as integrated water vapor anomal, we first compute the climatologies of each of these variables first. Specifically, we stream the monthly averaged dataset of our variables from MERRA-2 and further average across all years. Since the original Wille (2021) catalog on which ours is based goes from 1980 to 2022, we will be computing monthly averages over this range.

:::{note}
Either we can run the next cell to create the climatology dataset, which should take about 20 minutes to do. Or, you could run this just once, save it, and just load it up from disk for future runs. If you don't want to wait, we've already stored it in the `outputs` folder, so you can skip this cell and run the next one.
:::

In [ ]:
# grab granules for monthly data
granule_lst = earthaccess.search_data(doi=data_dois['climatology'], 
                                  temporal=('1980-01-01', '2022-12-31'))
# authenticate with earthdata
earthaccess.login()

# open pointers to nc4 stored in AWS S3 buckets
pointers = earthaccess.open(granule_lst, show_progress=False)

monthly_means = xr.open_mfdataset(pointers)
monthly_means = monthly_means[['T2M', 'TQV']]
climatology_ds = monthly_means.groupby(monthly_means.time.dt.month).mean().compute()
climatology_ds = climatology_ds.assign_coords(lat=climatology_ds.lat.round(5), lon=climatology_ds.lon.round(5))
climatology_ds.to_netcdf('../outputs/data_products/climatology.nc4')

In [ ]:
# or run this!
#climatology_ds = xr.load_dataset('../outputs/data_products/climatology.nc4')

We will be streaming quantities from a MERRA-2 dataset and extracting summary statistics on a storm-by-storm basis. We will be executing this loop in parallel fashion. ARs that last longer tend to be more costly to process; to ensure each worker receives roughly the same workload, we sort the storms by their landfalling duration.

In [ ]:
sorted_storms = storms.sort_values(by='duration', ascending=False)

We will want to log the progress of our computation. This is especially useful since `jupyter-keepalive` does not continuously print cell output upon exiting the notebook. A logger allows us to nonetheless track the progress of the workflow. Let's set up a logger here.

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler('../outputs/logs/dataset_construction.log')
    ]
)
logging.info("Starting notebook background execution...")

#### Integrated Water Vapor, 2m-Temperatures, Sea-Level Pressure

**Dataset**: [MERRA-2 inst1_2d_asm_Nx: 2d,1-Hourly,Instantaneous,Single-Level,Assimilation,Single-Level Diagnostics V5.12.4 (M2I1NXASM)](https://disc.gsfc.nasa.gov/datasets/M2I1NXASM_5.12.4/summary?keywords=10.5067%2F3Z173KIE2TPD)

This includes IWV, IWV anomaly, 2m-temperature, 2m-temperature anomaly, and sea level pressure quantities. Specifically, we compute, underneath the spatiotemporal footprint of the AR:
+ maximum 2m-temperature anomaly over the AIS (`max_T2M_anomaly_ais`)
+ maximum integrated water vapor over the AIS (`max_IWV_ais`)
+ maximum integrated water vapor anomaly overt the AIS (`max_IWV_anomaly_ais`)
+ maximum sea-level pressure gradient over ocean, at time of first landfall (`max_ocean_SLP_gradient`)

In our workflow, we split the storms into chunks of size 15, and then split those chunks into batches of size 10. For each batch of 15 chunks, we parallelize across our 15 workers so each worker processes a single chunk. To process a single chunk, a worker will go through each storm and execute a function called `compute_summaries`. `compute_summaries` computes scalar summaries of all desired variables from the particular MERRA-2 dataset we are
working with. 

To indicate which variables and which scalar summaries you would like, you must pass in a dictionary of dictionaries which we will call `func_vars_dict`. The keys of the outer dictionary are the variable names in the MERRA-2 dataset we'd like to compute aggregates of, and the values are themselves dictionaries where each item represents some desired aggregation function of that variable. The provided keys are the names of that scalar summary that will appear in the resulting dataset. 

After workers process their chunks and the batch is complete, we empty the Ray object store and move onto the next batch so as to reduce the memory footprint.

In [ ]:
num_cpus = NUM_CPUS

# initialize ray cluster
ray.init(num_cpus=num_cpus, logging_level='ERROR', 
         _metrics_export_port=-1, include_dashboard=False, 
         log_to_driver=False, runtime_env={'py_modules': [artools.attribute_utils, artools.loading_utils]})

# put the climatologies and cell areas in the ray object store
climatology_ref = ray.put(climatology_ds)
cell_areas_ref = ray.put(cell_areas)
# prevents parallel open requests from being made to NASA servers
gatekeeper = EarthdataGatekeeper.remote()

In [ ]:
func_vars_dict = {'T2M': {'max_T2M_anomaly_ais': lambda storm_da, var_da, area_da: 
                              compute_max_intensity(storm_da, var_da, area_da, ais_mask)},
                  'TQV': {'max_IWV_ais': lambda storm_da, var_da, area_da: 
                              compute_max_intensity(storm_da, var_da, area_da, ais_mask),
                          'max_IWV_anomaly_ais': lambda storm_da, var_da, area_da:
                              compute_max_intensity(storm_da, var_da, area_da, ais_mask)},
                  'SLP': {'max_ocean_SLP_gradient': lambda storm_da, var_da, area_da: 
                              compute_max_SLPgrad(storm_da, var_da, area_da, ais_mask)}}

func_vars_dict_ref = ray.put(func_vars_dict)

data_doi = data_dois['T2M_TQV_SLP']
# list with the computed results
results_T2M_TQV_SLP = []

chunk_size = CHUNK_SIZE
# split dataset of storms into chunks
chunks = [sorted_storms[i:i + chunk_size] for i in range(0, sorted_storms.shape[0], chunk_size)]
# number of chunks, i.e. tasks, to complete
batch_size = BATCH_SIZE

logging.info(f"Starting T2M_TQV_SLP computations in batches...")

for i in range(0, len(chunks), batch_size):
    batch_chunks = chunks[i:i + batch_size]
    
    # submit only this batch of tasks to ray
    batch_refs = [
        compute_chunk_summaries.remote(
            chunk,
            func_vars_dict_ref,
            cell_areas_ref,
            data_doi,
            gatekeeper=gatekeeper,
            climatology_ds=climatology_ref
        ) for chunk in batch_chunks
    ]
    
    # gather this batch's results and immediately clean up memory
    for j, ref in enumerate(batch_refs):
        results = ray.get(ref)
        results_T2M_TQV_SLP.extend(results)
        
        # clean up the reference immediately so ray empties the object store
        batch_refs[j] = None
        
    logging.info(f"T2M_TQV_SLP: Finished up to chunk {min(i + batch_size, len(chunks))}/{len(chunks)}")

logging.info("T2M_TQV_SLP computations complete.")

In [ ]:
# construct and save dataframe
labels = [key for quantity_dict in func_vars_dict.values() for key in quantity_dict.keys()]
df_T2M_TQV_SLP = pd.DataFrame(results_T2M_TQV_SLP, columns=labels, index=sorted_storms.index)
df_T2M_TQV_SLP.to_csv(OUTPUT_DIR + '/results_T2M_TQV_SLP.csv')
logging.info("T2M_TQV_SLP results saved!")

In [ ]:
ray.shutdown()

#### Poleward Integrated Vapor Transport

**Dataset**: [MERRA-2 tavg1_2d_int_Nx: 2d,1-Hourly,Time-Averaged,Single-Level,Assimilation,Vertically Integrated Diagnostics V5.12.4 (M2T1NXINT)](https://disc.gsfc.nasa.gov/datasets/M2T1NXINT_5.12.4/summary?keywords=vflxqv)

Next, we compute poleward integrated vapor transport quantities, or vIVT quantities. Specifically we compute the following, again underneath the footprint of the AR:
+ maximum poleward integrated vapor transport over the AIS (`max_vIVT_ais`)


The computation is very similar to the above, using the same aggregation functions, passed into the overall `compute_summaries` function. However, this time we aren't computing any anomalies so no need to pass in any climatologies, and we set the `half_hour` flag to true, since this dataset in MERRA-2 is hourly on the half-hour.

In [ ]:
num_cpus = NUM_CPUS

ray.init(num_cpus=num_cpus, logging_level='ERROR', 
         _metrics_export_port=-1, include_dashboard=False, 
         log_to_driver=False, runtime_env={'py_modules': [artools.attribute_utils, artools.loading_utils]})

climatology_ref = ray.put(climatology_ds)
cell_areas_ref = ray.put(cell_areas)
gatekeeper = EarthdataGatekeeper.remote()

In [ ]:
func_vars_dict = {'VFLXQV': {'max_vIVT_ais': lambda storm_da, var_da, area_da: 
                                 compute_max_intensity(storm_da, -var_da, area_da, ais_mask)}}

data_doi = data_dois['VFLXQV_PRECIP']
func_vars_dict_ref = ray.put(func_vars_dict)

In [ ]:
results_VFLXQV = []

chunk_size = CHUNK_SIZE
chunks = [sorted_storms[i:i + chunk_size] for i in range(0, sorted_storms.shape[0], chunk_size)]

batch_size = BATCH_SIZE

logging.info(f"Starting VFLXQV computations in batches...")

for i in range(0, len(chunks), batch_size):
    batch_chunks = chunks[i:i+batch_size]
    
    batch_refs = [
        compute_chunk_summaries.remote(
            chunk,
            func_vars_dict_ref,
            cell_areas_ref,
            data_doi,
            gatekeeper=gatekeeper,
            half_hour=True
        ) for chunk in batch_chunks
    ]
    for j, ref in enumerate(batch_refs):
        results = ray.get(ref)
        results_VFLXQV.extend(results)
        
        batch_refs[j] = None 
        
    logging.info(f"VFLXQV: Finished up to chunk {min(i+batch_size, len(chunks))}/{len(chunks)}")

logging.info("VFLXQV computations complete.")

In [ ]:
labels = [key for quantity_dict in func_vars_dict.values() for key in quantity_dict.keys()]
df_VFLXQV = pd.DataFrame(results_VFLXQV, columns=labels, index=sorted_storms.index)
df_VFLXQV.to_csv(OUTPUT_DIR + '/results_VFLXQV.csv')
logging.info("VFLXQV results saved!")

In [ ]:
ray.shutdown()

#### Snowfall and Rainfall

**Dataset**: [MERRA-2 tavg1_2d_int_Nx: 2d,1-Hourly,Time-Averaged,Single-Level,Assimilation,Vertically Integrated Diagnostics V5.12.4 (M2T1NXINT)](https://disc.gsfc.nasa.gov/datasets/M2T1NXINT_5.12.4/summary?keywords=vflxqv)

Snowfall and rainfall come from the same MERRA-2 dataset as poleward integrated vapor transport. However, we treat them separately because precipitation quantities are computed not on the spatiotemporal footprint of the mask as the previous quantities, but an augmented mask where even if the AR has left a pixel at a particular time, that pixel is still considered part of the AR footprint for the next 24 hours. Because of this extra preprocessing, we pass a `precip_func` function, the aggregation function we wish to use, into a summary computation function particular to this case (`compute_precip_summaries`).

Specifically, we wish to compute the following precipitation aggregates for each storm:
+ cumulative snowfall over the AIS (`cumulative_snowfall_ais`)
+ cumulative rainfall over the AIS (`cumulative_rainfall_ais`)

In [ ]:
num_cpus = NUM_CPUS

ray.init(num_cpus=num_cpus, logging_level='ERROR', 
         _metrics_export_port=-1, include_dashboard=False, 
         log_to_driver=False, runtime_env={'py_modules': [artools.attribute_utils, artools.loading_utils]})

climatology_ref = ray.put(climatology_ds)
cell_areas_ref = ray.put(cell_areas)
gatekeeper = EarthdataGatekeeper.remote()

In [ ]:
data_doi = data_dois['VFLXQV_PRECIP']

# storing the aggregation function for precip in the ray store
def precip_func(storm_da, var_da, area_da):
    return compute_cumulative(storm_da, var_da, area_da, ais_mask)
precip_func_ref = ray.put(precip_func)

In [ ]:
results_precip = []

chunk_size = CHUNK_SIZE
chunks = [sorted_storms[i:i + chunk_size] for i in range(0, sorted_storms.shape[0], chunk_size)]

batch_size = BATCH_SIZE

logging.info(f"Starting Precipitation computations in batches...")

for i in range(0, len(chunks), batch_size):
    batch_chunks = chunks[i:i + batch_size]
    
    # use special compute chunk summaries function for precip data
    batch_refs = [
        compute_precip_chunk_summaries.remote(
            chunk, 
            cell_areas_ref, 
            precip_func_ref,
            data_doi,
            gatekeeper=gatekeeper
        ) for chunk in batch_chunks
    ]
    
    for j, ref in enumerate(batch_refs):
        results = ray.get(ref)
        results_precip.extend(results)

        batch_refs[j] = None
        
    logging.info(f"Precipitation: Finished up to chunk {min(i + batch_size, len(chunks))}/{len(chunks)}")

logging.info("Precipitation computations complete.")

In [ ]:
labels = ['cumulative_rainfall_ais', 'cumulative_snowfall_ais']
df_precip = pd.DataFrame(results_precip, columns=labels, index=sorted_storms.index)
df_precip.to_csv(OUTPUT_DIR + '/results_precip.csv')
logging.info("Precip results saved!")

In [ ]:
ray.shutdown()

#### 850 hPa Poleward Wind

**Dataset**: [MERRA-2 tavg1_2d_slv_Nx: 2d,1-Hourly,Time-Averaged,Single-Level,Assimilation,Single-Level Diagnostics V5.12.4 (M2T1NXSLV)](https://disc.gsfc.nasa.gov/datasets/M2T1NXSLV_5.12.4/summary?keywords=10.5067%2FVJAFPLI1CSIV)

Now, we compute aggregates of the 850 hPa poleward wind, at the time of first landfall, under the portion of the AR footprint which is still over the ocean (`max_landfalling_v850hPa`). This is partially because 850 hPa wind is null on the continent, and this gives us some notion of how windy conditions were at landfall. To account for this, we use an alternative aggregation function in the `artools` package: `compute_max_landfalling_wind`.

In [ ]:
num_cpus = NUM_CPUS

ray.init(num_cpus=num_cpus, logging_level='ERROR', 
         _metrics_export_port=-1, include_dashboard=False, 
         log_to_driver=False, runtime_env={'py_modules': [artools.attribute_utils, artools.loading_utils]})

climatology_ref = ray.put(climatology_ds)
cell_areas_ref = ray.put(cell_areas)
gatekeeper = EarthdataGatekeeper.remote()

In [ ]:
func_vars_dict = {'V850': {'max_landfalling_v850hPa': lambda storm_da, var_da, area_da: 
                           compute_max_landfalling_wind(storm_da, -var_da, area_da, ais_mask)}}
data_doi = data_dois['V850']
results_V850 = []

func_vars_dict_ref = ray.put(func_vars_dict)

chunk_size = CHUNK_SIZE
chunks = [sorted_storms[i:i + chunk_size] for i in range(0, sorted_storms.shape[0], chunk_size)]

batch_size = BATCH_SIZE

logging.info(f"Starting V850 Wind computations in batches...")

for i in range(0, len(chunks), batch_size):
    batch_chunks = chunks[i:i + batch_size]
    
    batch_refs = [
        compute_chunk_summaries.remote(
            chunk,
            func_vars_dict_ref,
            cell_areas_ref,
            data_doi,
            gatekeeper=gatekeeper,
            half_hour=True
        ) for chunk in batch_chunks
    ]
    
    for j, ref in enumerate(batch_refs):
        results = ray.get(ref)
        results_V850.extend(results)
        batch_refs[j] = None
        
    logging.info(f"V850 Wind: Finished up to chunk {min(i + batch_size, len(chunks))}/{len(chunks)}")

logging.info("V850 Wind computations complete.")

In [ ]:
labels = [key for quantity_dict in func_vars_dict.values() for key in quantity_dict.keys()]
df_wind = pd.DataFrame(results_V850, columns=labels, index=sorted_storms.index)
df_wind.to_csv(OUTPUT_DIR + '/results_V850.csv')
logging.info("V850 wind results saved!")

In [ ]:
ray.shutdown()

#### Omega

**Dataset**: [MERRA-2 inst3_3d_asm_Np: 3d,3-Hourly,Instantaneous,Pressure-Level,Assimilation,Assimilated Meteorological Fields V5.12.4 (M2I3NPASM)](https://disc.gsfc.nasa.gov/datasets/M2I3NPASM_5.12.4/summary?keywords=omega)

Finally, we compute omega, or some notion of lifting in the storm. This is computed by first finding the portion of the AR footprint over the ice sheet at the time of first landfall. At these points, we find the minimum omega over all atmospheric levels, and then aggregate using a spatial average over that landfalling portion of the AR (`avg_landfalling_minomega`). To compute this, like for the last quantity, we use a particular aggregation function in the `artools` package called `compute_avg_landfalling_minomega`.

In [ ]:
num_cpus = NUM_CPUS

ray.init(num_cpus=num_cpus, logging_level='ERROR', 
         _metrics_export_port=-1, include_dashboard=False, 
         log_to_driver=False, runtime_env={'py_modules': [artools.attribute_utils, artools.loading_utils]})

climatology_ref = ray.put(climatology_ds)
cell_areas_ref = ray.put(cell_areas)
gatekeeper = EarthdataGatekeeper.remote()

In [ ]:
func_vars_dict = {'OMEGA': {'avg_landfalling_minomega': lambda storm_da, var_da, area_da: 
                             compute_avg_landfalling_minomega(storm_da, var_da, area_da, ais_mask)}}
data_doi = data_dois['OMEGA']
results_omega = []

func_vars_dict_ref = ray.put(func_vars_dict)

chunk_size = CHUNK_SIZE
chunks = [sorted_storms[i:i + chunk_size] for i in range(0, sorted_storms.shape[0], chunk_size)]

batch_size = BATCH_SIZE

logging.info(f"Starting OMEGA computations in batches...")

for i in range(0, len(chunks), batch_size):
    batch_chunks = chunks[i:i + batch_size]

    batch_refs = [
        compute_chunk_summaries.remote(
            chunk,
            func_vars_dict_ref,
            cell_areas_ref,
            data_doi,
            gatekeeper=gatekeeper
        ) for chunk in batch_chunks
    ]
    
    for j, ref in enumerate(batch_refs):
        results = ray.get(ref)
        results_omega.extend(results)
        batch_refs[j] = None
        
    logging.info(f"OMEGA: Finished up to chunk {min(i + batch_size, len(chunks))}/{len(chunks)}")

logging.info("OMEGA computations complete.")

In [ ]:
labels = [key for quantity_dict in func_vars_dict.values() for key in quantity_dict.keys()]
df_omega = pd.DataFrame(results_omega, columns=labels, index=sorted_storms.index)
df_omega.to_csv(OUTPUT_DIR + '/results_omega.csv')
logging.info("OMEGA wind results saved!")

In [ ]:
ray.shutdown()

### Concatenating at Last

Let's load up all of our intermediate dataframes we created along the way and concatenate them together into one big dataset of Antarctic ARs identified from 1980-2022.

In [ ]:
storms_setting = pd.read_hdf('../outputs/data_products/storms_setting.h5')
storms_t2m_tqv_slp = pd.read_csv('../outputs/data_products/results_T2M_TQV_SLP.csv', index_col='cluster')
storms_precip = pd.read_csv('../outputs/data_products/results_precip.csv', index_col='cluster')
storms_omega = pd.read_csv('../outputs/data_products/results_omega.csv', index_col='cluster')
storms_wind = pd.read_csv('../outputs/data_products/results_V850.csv', index_col='cluster')
storms_ivt = pd.read_csv('../outputs/data_products/results_VFLXQV.csv', index_col='cluster')

In [ ]:
full_df = pd.concat([storms_setting,
                     storms_t2m_tqv_slp,
                     storms_precip,
                     storms_omega,
                     storms_wind,
                     storms_ivt], axis=1)

Finally, let's display the first 10 rows of our catalog.

In [ ]:
display_catalog(full_df, 10)

Finally, let's save our full dataframe output.

In [ ]:
full_df.to_hdf('../outputs/data_products/full_AR_df.h5', key='df')